# Notebook 02 — Computing All 20 Metacognitive Measures

`metasignal.stdpy.compute_all_measures(stim, resp, conf, n_ratings)` returns a NumPy array of 20 values:

| Index | Measure | Type |
|-------|---------|------|
| 0 | meta-d' | Absolute (MLE) |
| 1–4 | AUC2, Gamma, Phi, DeltaConf | Absolute |
| 5–9 | M-Ratio, AUC2-Ratio, Gamma-Ratio, Phi-Ratio, DeltaConf-Ratio | Ratio-normalised |
| 10–14 | M-Diff, AUC2-Diff, Gamma-Diff, Phi-Diff, DeltaConf-Diff | Difference-normalised |
| 15 | meta-noise | Meta-noise model |
| 16 | meta-uncertainty | Meta-uncertainty |
| 17–19 | d', Criterion, Confidence | SDT basics |

We compute these for every subject in every dataset and save to `precomputed/`.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


## Preprocessing (inline)

In [ ]:
ACC_LO, ACC_HI, MAX_PROP = 0.60, 0.95, 0.85

# ── exclusion thresholds (match MATLAB step2_preprocessData.m) ──
ACC_LO, ACC_HI, MAX_PROP = 0.60, 0.95, 0.85

def _exclude(acc, pr, pc):
    """Return True if subject should be excluded."""
    return acc < ACC_LO or acc > ACC_HI or pr > MAX_PROP or pc > MAX_PROP

def preprocess_haddara():
    """Load Haddara 2022 Expt2. n_ratings=4. Returns list of subject dicts."""
    df = pd.read_csv(os.path.join(DATA, 'data_Haddara_2022_Expt2.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim = g['Stimulus'].to_numpy(float)
        resp = g['Response'].to_numpy(float)
        conf = g['Confidence'].to_numpy(float)
        day  = g['Day'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        pc  = np.max(np.unique(conf, return_counts=True)[1]) / len(conf)
        if _exclude(acc, pr, pc): continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf, day=day, n_ratings=4))
    return subjects

def preprocess_maniscalco():
    """Load Maniscalco 2017 Expt1. NaN responses counted as incorrect. n_ratings=4."""
    df = pd.read_csv(os.path.join(DATA, 'data_Maniscalco_2017_expt1.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        stim_all = g['Stimulus'].to_numpy(float)
        resp_all = g['Response'].to_numpy(float)
        conf_all = g['Confidence'].to_numpy(float)
        # MATLAB: NaN responses count as incorrect
        correct = np.where(np.isnan(resp_all), 0.0, (stim_all == resp_all).astype(float))
        acc = np.mean(correct)
        pr  = np.max(np.bincount(resp_all[~np.isnan(resp_all)].astype(int))) / len(resp_all)
        valid_c = conf_all[~np.isnan(conf_all)]
        pc  = np.max(np.bincount(valid_c.astype(int))) / len(conf_all)
        if _exclude(acc, pr, pc): continue
        ok = ~(np.isnan(stim_all) | np.isnan(resp_all) | np.isnan(conf_all))
        subjects.append(dict(sid=sid, stim=stim_all[ok], resp=resp_all[ok],
                             conf=conf_all[ok], n_ratings=4))
    return subjects

def preprocess_shekhar():
    """Load Shekhar 2021. Continuous conf (50–100) binned to n_ratings=6 levels."""
    df = pd.read_csv(os.path.join(DATA, 'data_Shekhar_2021.csv'))
    n_ratings = 6
    edges = np.linspace(50, 100, n_ratings + 1)
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        stim     = g['Stimulus'].to_numpy(float)
        resp     = g['Response'].to_numpy(float)
        conf_raw = g['Confidence'].to_numpy(float)
        contrast = g['Contrast'].to_numpy(float)
        conf = np.clip(np.digitize(conf_raw, edges, right=True), 1, n_ratings)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        pc  = np.max(np.unique(conf, return_counts=True)[1]) / len(conf)
        if _exclude(acc, pr, pc): continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             contrast=contrast, n_ratings=n_ratings))
    return subjects

def preprocess_rouault(expt=1):
    """Load Rouault 2018 Expt1 or Expt2. Stereotypy on raw conf; transform after. n_ratings=6."""
    df = pd.read_csv(os.path.join(DATA, f'data_Rouault_2018_Expt{expt}.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim     = g['Stimulus'].to_numpy(float)
        resp     = g['Response'].to_numpy(float)
        conf_raw = g['Confidence'].to_numpy(float)
        dotdiff  = g['DotDiff'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.bincount(resp.astype(int))) / len(resp)
        # MATLAB: stereotypy on RAW conf (1-11) BEFORE transformation
        pc  = np.max(np.bincount(conf_raw.astype(int))) / len(conf_raw)
        if _exclude(acc, pr, pc): continue
        conf = np.clip(conf_raw - 5, 1, None) if expt == 1 else conf_raw.copy()
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             contrast=dotdiff, n_ratings=6))
    return subjects

def preprocess_locke():
    """Load Locke 2020 (test trials only). n_ratings=2."""
    df = pd.read_csv(os.path.join(DATA, 'data_Locke_2020.csv'))
    df = df[df['Training'] == 0].copy()
    df['Confidence'] = df['Confidence'] + 1   # 0/1 → 1/2
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim      = g['Stimulus'].to_numpy(float)
        resp      = g['Response'].to_numpy(float)
        conf      = g['Confidence'].to_numpy(float)
        condition = g['Condition'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        if acc < ACC_LO or acc > ACC_HI or pr > MAX_PROP: continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             condition=condition, n_ratings=2))
    return subjects

print("Preprocessing functions defined.")


In [ ]:
MEASURE_NAMES = [
    "meta-d'", "AUC2", "Gamma", "Phi", "DeltaConf",
    "M-Ratio", "AUC2-Ratio", "Gamma-Ratio", "Phi-Ratio", "DeltaConf-Ratio",
    "M-Diff",  "AUC2-Diff",  "Gamma-Diff",  "Phi-Diff",  "DeltaConf-Diff",
    "meta-noise", "meta-uncertainty",
    "d'", "Criterion", "Confidence",
]
N_MEAS = 20


## Load datasets

In [ ]:
haddara    = preprocess_haddara()
maniscalco = preprocess_maniscalco()
shekhar    = preprocess_shekhar()
rouault1   = preprocess_rouault(1)
rouault2   = preprocess_rouault(2)
locke      = preprocess_locke()
print(f"Subjects: Ha={len(haddara)} Ma={len(maniscalco)} Sh={len(shekhar)} "
      f"R1={len(rouault1)} R2={len(rouault2)} Lo={len(locke)}")


## Compute measures — simple datasets (Haddara & Maniscalco)

We load precomputed files if available (MLE fitting for meta-d' is slow).

In [ ]:
def compute_raw(subjects, label):
    path = os.path.join(OUT, f'{label}_mle.npz')
    if os.path.exists(path):
        data = np.load(path)
        print(f"Loaded {label}: {data['raw'].shape}")
        return data['raw']
    print(f"Computing {label}... (this may take a few minutes for meta-d')")
    raw = np.array([compute_all_measures(s['stim'],s['resp'],s['conf'],s['n_ratings'])
                    for s in subjects])
    np.savez(path, raw=raw)
    print(f"  Saved {label}: {raw.shape}")
    return raw

ha_raw = compute_raw(haddara,    'haddara')
ma_raw = compute_raw(maniscalco, 'maniscalco')


## Compute measures — difficulty datasets (Shekhar, Rouault1, Rouault2)

For each subject we compute measures separately per difficulty level.

In [ ]:
def compute_difficulty_shekhar(subjects):
    """Returns (n_sub, 3, 20) — one row per contrast [1, 2, 3]."""
    path = os.path.join(OUT, 'shekhar_mle.npz')
    if os.path.exists(path):
        data = np.load(path); print("Loaded shekhar diff:", data['diff'].shape); return data['diff']
    print("Computing Shekhar difficulty measures...")
    out = []
    for s in subjects:
        row = np.full((3, N_MEAS), np.nan)
        for ci, c in enumerate([1,2,3]):
            mask = s['contrast'] == c
            if mask.sum() >= 10:
                row[ci] = compute_all_measures(s['stim'][mask], s['resp'][mask],
                                               s['conf'][mask], s['n_ratings'])
        out.append(row)
    arr = np.array(out)
    np.savez(path, diff=arr)
    return arr

def compute_difficulty_rouault(subjects, label):
    """Returns (n_sub, 2, 20) — low and high contrast halves."""
    path = os.path.join(OUT, f'{label}_mle.npz')
    if os.path.exists(path):
        data = np.load(path); print(f"Loaded {label} diff:", data['diff'].shape); return data['diff']
    print(f"Computing {label} difficulty measures...")
    out = []
    for s in subjects:
        med = np.median(s['contrast'])
        row = np.full((2, N_MEAS), np.nan)
        for li, mask in enumerate([s['contrast'] <= med, s['contrast'] > med]):
            if mask.sum() >= 10:
                row[li] = compute_all_measures(s['stim'][mask], s['resp'][mask],
                                               s['conf'][mask], s['n_ratings'])
        out.append(row)
    arr = np.array(out)
    np.savez(path, diff=arr)
    return arr

sh_diff  = compute_difficulty_shekhar(shekhar)
r1_diff  = compute_difficulty_rouault(rouault1, 'rouault1')
r2_diff  = compute_difficulty_rouault(rouault2, 'rouault2')


## Compute measures — Locke (7 response-bias conditions)

In [ ]:
def compute_locke_rb(subjects):
    path = os.path.join(OUT, 'locke_mle.npz')
    if os.path.exists(path):
        data = np.load(path); print("Loaded locke rb:", data['rb'].shape); return data['rb']
    out = []
    for s in subjects:
        row = np.full((7, N_MEAS), np.nan)
        for ci, cond in enumerate(range(1,8)):
            mask = s['condition'] == cond
            if mask.sum() >= 5:
                row[ci] = compute_all_measures(s['stim'][mask], s['resp'][mask],
                                               s['conf'][mask], s['n_ratings'])
        out.append(row)
    arr = np.array(out)
    np.savez(path, rb=arr)
    return arr

lo_rb = compute_locke_rb(locke)


## Summary statistics

In [ ]:
md_lbl = "Mean meta-d'"
print(f"{'Dataset':<12} {'Shape':<20} {md_lbl:>14} {'Mean AUC2':>12}")
print("-"*60)
for label, arr in [('Haddara', ha_raw), ('Maniscalco', ma_raw)]:
    md_m  = np.nanmean(arr[:,0])
    auc_m = np.nanmean(arr[:,1])
    print(f"{label:<12} {str(arr.shape):<20} {md_m:>14.3f} {auc_m:>12.3f}")
for label, arr in [('Shekhar', sh_diff), ('Rouault1', r1_diff), ('Rouault2', r2_diff)]:
    avg = np.nanmean(arr, axis=1)   # average over difficulty levels
    print(f"{label:<12} {str(arr.shape):<20} {np.nanmean(avg[:,0]):>14.3f} {np.nanmean(avg[:,1]):>12.3f}")
